# BKQuiz · SFT trực tiếp Qwen3-1.7B từ các trang đã hoàn tất
**361 câu /181trang; train281, validation40, test40.** Loại toàn bộ trang còn cần LLM/VLM/duyệt thủ công.
Không chạy teacher7B, OCR, VLM hoặc sinh thêm lời giải. Chọn **Runtime → Change runtime type → T4 GPU**.

ZIP đã chứa dataset tokenized, tokenizer và adapter CPT27 đúng hash. Chỉ base Qwen3-1.7B được tải từ HuggingFace
theo revision cố định. SFT train tiếp chính adapter CPT27,1epoch; output là adapter hoàn chỉnh CPT+SFT.
T4 dùng FP16, BF16 chỉ dùng nếu GPU hỗ trợ native. Người dùng tự chạy GPU.

Dữ liệu huấn luyện biên soạn Q&A nguồn thành quiz JSON, có nhiều lời giải ngắn dẫn đáp án nguồn.
Chưa phải bộ dạy đọc lý thuyết rồi tự tạo câu mới và lời giải sâu. Đáp án nguồn không phải expert gold.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

In [ ]:
import hashlib, json, os, sys, subprocess, shutil, zipfile, stat
from pathlib import PurePosixPath
DRIVE_ROOT=Path('/content/drive/MyDrive/BKQuiz_CPT/SFT_COMPLETED_NATIVE_20261025')
ZIP_PATH=DRIVE_ROOT/'bkquiz_completed_sft_20261025.zip'
EXPECTED_ZIP_SHA256='db56e162ae65190651ecd03431395d0f2ce4ebcfe83cc134c75dcd48dd74e57d'
RUN_NAME='sft_completed_native_20261025_001'
RESUME_TRAIN=True
RUN_SMOKE=False  # tùy chọn kiểm tra2steps; không dùng adapter smoke làm kết quả cuối
EVALUATE_TEST=False  # test được giữ riêng; chỉ bật khi cần đánh giá cuối
assert ZIP_PATH.is_file(), f'Chưa tìm thấy {ZIP_PATH}'

In [ ]:
def file_sha(path):
    with Path(path).open('rb') as stream: return hashlib.file_digest(stream,'sha256').hexdigest()
LOCAL_ZIP=Path('/content/bkquiz_completed_sft_20261025.zip')
shutil.copyfile(ZIP_PATH,LOCAL_ZIP)
assert file_sha(LOCAL_ZIP)==EXPECTED_ZIP_SHA256, 'ZIP/notebook không cùng bản; tải lại đúng hai tệp'
WORK=Path('/content/bkquiz_completed_sft_'+EXPECTED_ZIP_SHA256[:12])
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    infos=archive.infolist(); names=[entry.filename for entry in infos]
    assert len(names)==len(set(names)) and len(names)<250, 'ZIP trùng hoặc quá nhiều file'
    assert sum(i.file_size for i in infos)<512*1024*1024, 'ZIP vượt ngân sách'
    for entry in infos:
        name=entry.filename; rel=PurePosixPath(name)
        assert not rel.is_absolute() and '..' not in rel.parts and '\\' not in name and ':' not in name
        assert stat.S_IFMT(entry.external_attr>>16)!=stat.S_IFLNK and entry.file_size<128*1024*1024
        assert (WORK/name).resolve().is_relative_to(WORK.resolve())
    manifest=json.loads(archive.read('manifest.json'))
    assert set(names)=={'manifest.json',*manifest['files_sha256']}, 'File ZIP không khớp registry'
    archive.extractall(WORK)
for name,wanted in manifest['files_sha256'].items():
    assert file_sha(WORK/name)==wanted, 'File bị thay đổi: '+name
assert manifest['schema']=='bkquiz-completed-sft-v1'
assert manifest['source_policy']=='completed_native_pages_only' and manifest['no_pending_llm_vlm_manual_pages']
print('Đã kiểm tra ZIP và mọi file. Split:',manifest['counts'])
print('CPT adapter có sẵn trong:',WORK/'assets/cpt_adapter')

## Môi trường và kiểm tra dữ liệu
Dùng CUDA torch có sẵn trong Colab và venv riêng. Các thư viện còn lại được khóa phiên bản.
Kiểm tra mọi hash, các trang bị loại, split và mask assistant trước khi tải trọng số model.

In [ ]:
VENV=Path('/content/bkquiz_completed_sft_venv')
PYTHON=VENV/'bin/python'
if not PYTHON.exists():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
RUNTIME=WORK/'runtime'
# Teacher không dùng trong notebook này; tắt optional vision/audio imports
# để tránh ABI của thư viện hệ thống không liên quan với torch Colab.
(RUNTIME/'sitecustomize.py').write_text("import importlib.util\n_original=importlib.util.find_spec\ndef _find(name,*args,**kwargs):\n    if name.split('.')[0] in {'torchvision','torchaudio','librosa'}:return None\n    return _original(name,*args,**kwargs)\nimportlib.util.find_spec=_find\n",encoding='utf-8')
ENV=dict(os.environ,PYTHONPATH=str(RUNTIME),TOKENIZERS_PARALLELISM='false')
subprocess.run([str(PYTHON),'-m','pip','install','--quiet','-r',str(RUNTIME/'requirements.txt')],env=ENV,check=True)
subprocess.run([str(PYTHON),'-c','import torch; assert torch.cuda.is_available(), "Select GPU runtime"; print(torch.__version__,torch.cuda.get_device_properties(0))'],env=ENV,check=True)
def launch(*args):
    subprocess.run([str(PYTHON),'-u',str(RUNTIME/'train_sft.py'),*map(str,args)],env=ENV,cwd=str(RUNTIME),check=True)
launch('--data',WORK,'--mode','preflight')

## Huấn luyện
Mặc định1epoch, LR2e-5, batch1, accumulation8, khoảng36optimizersteps.
Lưu checkpoint mỗi5steps, validation để chọn checkpoint; không tự dùng test.
Giữ RUN_NAME và RESUME_TRAIN=True để tiếp tục cùng dữ liệu, code và thư viện.
Khi đổi dữ liệu/cấu hình/thư viện, chọn RUN_NAME mới. Thời gian train thực tế cần đo trên GPU của bạn.

In [ ]:
RUN_DIR=DRIVE_ROOT/'runs'/RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
if RUN_SMOKE:
    SMOKE=RUN_DIR/'smoke'
    args=['--data',WORK,'--output',SMOKE,'--mode','smoke']
    if RESUME_TRAIN and (SMOKE/'run_identity.json').is_file(): args+=['--resume','latest']
    launch(*args)
TRAIN=RUN_DIR/'train'
args=['--data',WORK,'--output',TRAIN,'--mode','train']
if RESUME_TRAIN and (TRAIN/'run_identity.json').is_file(): args+=['--resume','latest']
if EVALUATE_TEST: args+=['--evaluate-test']
launch(*args)

In [ ]:
from google.colab import files
REPORT=TRAIN/'completed.json'
assert REPORT.is_file(), 'Chưa train xong; giữ RUN_NAME để resume checkpoint'
result=json.loads(REPORT.read_text())
assert result['status']=='completed' and result['mode']=='train' and result['adapter_weights_changed']
print(json.dumps(result,ensure_ascii=False,indent=2))
OUTPUT_ZIP=TRAIN/result['bundle_name']
bundle_info=json.loads((TRAIN/'RESULT_BUNDLE.json').read_text())
assert file_sha(OUTPUT_ZIP)==bundle_info['sha256'], 'ZIP kết quả không khớp hash'
print('Adapter CPT+SFT:',TRAIN/'adapter')
print('Kết quả luôn lưu trên Drive:',OUTPUT_ZIP)
files.download(str(OUTPUT_ZIP))

## Dùng kết quả
`sft_completed_adapter.zip` chứa adapter đã tiếp tục từ CPT27, tokenizer, metrics và nguồn gốc.
Nạp adapter mới một lần lên đúng base Qwen3-1.7B; không chồng thêm adapter CPT cũ.
Đây là adapter PEFT, chưa phải full weights/GGUF. Chưa tự thay model đang phục vụ.
Nếu Colab bị ngắt, kết nối GPU rồi Run all lại; dùng checkpoint đầy đủ cùng RUN_NAME.
Nếu bị ngắt trước checkpoint đầu tiên, chọn RUN_NAME mới vì chưa có trạng thái optimizer để resume.